In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, hour, minute, second, when, concat_ws, lpad, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd

print("CREATING TIME DIMENSION")

gold_time_dim_path = "s3://travel-analytics-bronze/delta/gold/Dim_Time/"

# =============================================================
# STEP 1: GENERATE TIME DATA (24 hours, 60 minutes, 60 seconds)
# =============================================================
print("\nSTEP 1: Generating Time Data...")

# Create a list of all time combinations (24 * 60 * 60 = 86400 records)
time_data = []

for h in range(24):
    for m in range(60):
        for s in range(60):
            # Format time components
            hour_24 = h
            hour_12 = 12 if h % 12 == 0 else h % 12
            am_pm = "AM" if h < 12 else "PM"
            
            # Format full time string
            full_time = f"{h:02d}:{m:02d}:{s:02d}"
            
            # Create hour bucket (common time grouping)
            if h < 6:
                hour_bucket = "Late Night (0-5)"
            elif h < 12:
                hour_bucket = "Morning (6-11)"
            elif h < 17:
                hour_bucket = "Afternoon (12-16)"
            elif h < 21:
                hour_bucket = "Evening (17-20)"
            else:
                hour_bucket = "Night (21-23)"
            
            time_data.append((full_time, h, m, s, hour_24, hour_12, am_pm, hour_bucket))

# Create DataFrame from the generated data
time_schema = StructType([
    StructField("Full_Time", StringType(), False),
    StructField("Hour", IntegerType(), False),
    StructField("Minute", IntegerType(), False),
    StructField("Second", IntegerType(), False),
    StructField("Hour_24", IntegerType(), False),
    StructField("Hour_12", IntegerType(), False),
    StructField("AM_PM", StringType(), False),
    StructField("Hour_Bucket", StringType(), False)
])

time_df = spark.createDataFrame(time_data, schema=time_schema)

print(f"Generated {time_df.count():,} time records")

# =============================================================
# STEP 2: ADD SURROGATE KEY AND METADATA
# =============================================================
print("\nSTEP 2: Creating Dimension Structure")
dim_time_df = (
    time_df
    
    # Add surrogate key
    .withColumn("Time_Dim_SK", monotonically_increasing_id() + 1)

    # Reorder columns to match dimension model
    .select(
        col("Time_Dim_SK"),           # PK
        col("Full_Time"),
        col("Hour"),
        col("Minute"),
        col("Second"),
        col("Hour_24"),
        col("Hour_12"),
        col("AM_PM"),
        col("Hour_Bucket")
    )
)



CREATING TIME DIMENSION

STEP 1: Generating Time Data...
Generated 86,400 time records

STEP 2: Creating Dimension Structure


In [0]:
dim_time_df.display()

Time_Dim_SK,Full_Time,Hour,Minute,Second,Hour_24,Hour_12,AM_PM,Hour_Bucket
1,00:00:00,0,0,0,0,12,AM,Late Night (0-5)
2,00:00:01,0,0,1,0,12,AM,Late Night (0-5)
3,00:00:02,0,0,2,0,12,AM,Late Night (0-5)
4,00:00:03,0,0,3,0,12,AM,Late Night (0-5)
5,00:00:04,0,0,4,0,12,AM,Late Night (0-5)
6,00:00:05,0,0,5,0,12,AM,Late Night (0-5)
7,00:00:06,0,0,6,0,12,AM,Late Night (0-5)
8,00:00:07,0,0,7,0,12,AM,Late Night (0-5)
9,00:00:08,0,0,8,0,12,AM,Late Night (0-5)
10,00:00:09,0,0,9,0,12,AM,Late Night (0-5)


In [0]:
# =============================================================
# STEP 3: SAVE TO GOLD LAYER
# =============================================================
print("\nSTEP 3: Saving to Gold Layer...")
dim_time_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_time_dim_path)

print(f"Successfully created Dim_Time with {dim_time_df.count():,} records")


STEP 3: Saving to Gold Layer...
Successfully created Dim_Time with 86,400 records
